# NB5 — Visualizações e Análise Crítica dos Resultados (TCC)

**Pipeline de Predição de Crises Epilépticas a partir de EEG**

Este notebook carrega os CSVs gerados pelo NB4 e produz figuras prontas para o TCC:
- Comparação de janelas W (Etapa 1)
- Comparação de níveis de canais (Etapa 2)
- Comparação de modelos por dataset (Etapa 3)
- Desempenho por paciente (heatmap)
- Radar chart de métricas
- Diagnóstico do bug N_Ctx=1
- Proposta de melhorias (z-score + múltiplas seeds)

> **Pré-requisito:** rode NB1→NB4 antes e tenha `data/results/*.csv` disponíveis.

## 0. Imports e tema visual

In [ ]:
import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
import matplotlib.cm as cm

try:
    import seaborn as sns
    HAS_SNS = True
except ImportError:
    HAS_SNS = False
    print('Instale seaborn para figuras melhores: pip install seaborn')

warnings.filterwarnings('ignore')

# ── Tema visual ───────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 130,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'legend.fontsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})
if HAS_SNS:
    sns.set_palette('tab10')

# Paleta de datasets
DS_COLOR = {
    'CHBMIT':   '#2196F3',  # azul
    'Siena':    '#4CAF50',  # verde
    'SeizeIT2': '#FF9800',  # laranja
    'Mendeley': '#9C27B0',  # roxo
}

RES_DIR  = Path('data/results')
FIG_DIR  = Path('data/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

FEAT_DIR = Path('data/features')

print('Imports OK. Figuras serão salvas em:', FIG_DIR)

## 1. Carregamento dos resultados do NB4

In [ ]:
def load_csv(name):
    fp = RES_DIR / name
    if not fp.exists():
        print(f'  ⚠  {name} não encontrado — rode o NB4 primeiro.')
        return pd.DataFrame()
    df = pd.read_csv(fp)
    print(f'  {name}: {len(df)} linhas, {df["paciente"].nunique()} pacientes')
    return df

print('Carregando resultados...')
df_s1 = load_csv('stage1_windows.csv')
df_s2 = load_csv('stage2_levels.csv')
df_s3 = load_csv('stage3_models.csv')

# Sumário opcional
summary_path = RES_DIR / 'nb4_summary.json'
SUMMARY = {}
if summary_path.exists():
    with open(summary_path, encoding='utf-8') as f:
        SUMMARY = json.load(f)
    print(f'  Sumário: melhor nível={SUMMARY.get("best_level")}, '
          f'melhor modelo={SUMMARY.get("best_model_global")} '
          f'(AUC={SUMMARY.get("best_model_auc_global",0):.3f})')

---
## 2. Diagnóstico — Bug N_Ctx = 1

O NB1 garante `n_contextos > 2` por paciente, mas nas tabelas do NB4 aparecem
pacientes com `N_Ctx_Total = 1`.  

**Causa raiz:** `filter_window()` usa `t_inter >= -W_s` para selecionar janelas do
interictal. Se um contexto tem interictal mais curto que W_s (ex: contexto salvou
apenas 30 min mas W = 25 min e o `t_inter_from_end` desse contexto não alcança
−1140 s), suas janelas **somem** após o filtro → `set(ci)` perde o ctx_id dele.  

Resultado: `contexts = sorted(set(cp) | set(ci))` tem só 1 elemento → LOSO retorna
`[]` ou o `N_Ctx` das tabelas reporta 1.

**Correção proposta (célula 5 do NB4):** antes de fazer o LOSO, verificar contextos
presentes *tanto* em `cp` quanto em `ci`. Descartamos um contexto do fold somente
se ele não tiver dados em nenhuma das duas janelas.

In [ ]:
# ── Identifica pacientes com N_Ctx_Total suspeito (= 1) ──────────────────────
for df, nome in [(df_s1,'Etapa 1'),(df_s2,'Etapa 2'),(df_s3,'Etapa 3')]:
    if df.empty: continue
    n_ctx = df.groupby(['dataset','paciente'])['fold_ctx'].nunique()
    problema = n_ctx[n_ctx == 1]
    if len(problema):
        print(f'\n⚠  {nome} — pacientes com N_Ctx=1 (bug filter_window):')
        for (ds,pat), _ in problema.items():
            auc_meio = df[(df.dataset==ds)&(df.paciente==pat)]['auc'].mean()
            print(f'   {ds}/{pat}  AUC={auc_meio:.3f}  '
                  f'(fold único = sem leave-out real!)')
    else:
        print(f'\n✓  {nome} — nenhum paciente com N_Ctx=1')

print()
print('CORREÇÃO NO NB4 (em run_loso_patient, antes de chamar _run_loso_generic):')
print('''
    # Garante que LOSO só usa contextos com dados nas duas classes
    contexts_pre   = set(cp)
    contexts_inter = set(ci)
    contexts = sorted(contexts_pre & contexts_inter)  # interseção, não união
    if len(contexts) < 2:
        return []
''')

---
## 3. Etapa 1 — Comparação de janelas W

Janela escolhida por paciente via LOSO aninhado (sem vazamento). A figura abaixo
mostra a distribuição de AUC por janela e o número de vezes que cada janela foi
a preferida nos folds.

In [ ]:
if df_s1.empty:
    print('df_s1 vazio — pule esta célula.')
else:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle('Etapa 1 — Comparação de Janelas Pré-ictais W\n'
                 '(LOSO aninhado, sem vazamento, RF, nível máximo)', fontsize=13, y=1.01)

    # ── Subplot 1: boxplot AUC por janela ─────────────────────────────────────
    ax = axes[0]
    windows_order = sorted(df_s1['chosen_window'].unique(),
                           key=lambda x: int(x[1:]) if x[1:].isdigit() else 99)
    data_box = [df_s1[df_s1['chosen_window']==w]['auc'].dropna().values
                for w in windows_order]
    bp = ax.boxplot(data_box, labels=windows_order, patch_artist=True, notch=False,
                    medianprops={'color':'black','linewidth':2})
    colors_box = plt.cm.Blues(np.linspace(0.4, 0.85, len(windows_order)))
    for patch, color in zip(bp['boxes'], colors_box):
        patch.set_facecolor(color)
    ax.axhline(0.5, color='red', linestyle='--', linewidth=1, alpha=0.7, label='Aleatório (0.5)')
    ax.set_xlabel('Janela')
    ax.set_ylabel('AUC')
    ax.set_title('Distribuição de AUC por janela')
    ax.legend(fontsize=9)
    ax.set_ylim(0.3, 1.05)

    # ── Subplot 2: frequência de escolha por janela ───────────────────────────
    ax = axes[1]
    freq = df_s1['chosen_window'].value_counts().reindex(windows_order, fill_value=0)
    bars = ax.bar(freq.index, freq.values,
                  color=[colors_box[i] for i in range(len(windows_order))],
                  edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, freq.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                str(val), ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_xlabel('Janela')
    ax.set_ylabel('Nº de folds onde foi escolhida')
    ax.set_title('Frequência de escolha da janela')

    # ── Subplot 3: AUC médio ± std por janela + métricas ─────────────────────
    ax = axes[2]
    agg = df_s1.groupby('chosen_window').agg(
        auc_mean=('auc','mean'), auc_std=('auc','std'),
        sens_mean=('sensitivity','mean'),
        spec_mean=('specificity','mean')
    ).reindex(windows_order)

    x = np.arange(len(windows_order))
    w_bar = 0.25
    ax.bar(x - w_bar, agg['auc_mean'], width=w_bar, label='AUC',
           color='#2196F3', alpha=0.85)
    ax.bar(x, agg['sens_mean'], width=w_bar, label='Sensibilidade',
           color='#4CAF50', alpha=0.85)
    ax.bar(x + w_bar, agg['spec_mean'], width=w_bar, label='Especificidade',
           color='#FF9800', alpha=0.85)
    ax.errorbar(x - w_bar, agg['auc_mean'], yerr=agg['auc_std'],
                fmt='none', color='black', capsize=3, linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(windows_order)
    ax.set_xlabel('Janela')
    ax.set_ylabel('Valor médio')
    ax.set_title('AUC · Sens · Espec por janela')
    ax.legend(fontsize=9)
    ax.set_ylim(0, 1.1)
    ax.axhline(0.5, color='red', linestyle='--', linewidth=1, alpha=0.5)

    plt.tight_layout()
    fig.savefig(FIG_DIR / 'etapa1_janelas.png', bbox_inches='tight', dpi=150)
    plt.show()
    print(f'Figura salva em {FIG_DIR}/etapa1_janelas.png')

---
## 4. Etapa 2 — Comparação de níveis de canais

In [ ]:
if df_s2.empty:
    print('df_s2 vazio — pule esta célula.')
else:
    LEVELS = ['R0', 'R1', 'R2', 'R3', 'R4', 'R5']
    N_CH   = {'R0':2, 'R1':4, 'R2':11, 'R3':13, 'R4':15, 'R5':17}

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle('Etapa 2 — Impacto do Número de Canais (Nível R0→R5)\n'
                 '(janela consenso por paciente, RF)', fontsize=13, y=1.01)

    agg_s2 = (df_s2.groupby('level').agg(
                  auc_mean=('auc','mean'), auc_std=('auc','std'),
                  f1_mean=('f1','mean'),
                  sens_mean=('sensitivity','mean'),
                  spec_mean=('specificity','mean')
              ).reindex(LEVELS))

    # ── Subplot 1: linha AUC por nível, separado por dataset ──────────────────
    ax = axes[0]
    for ds, color in DS_COLOR.items():
        sub = df_s2[df_s2['dataset']==ds]
        if sub.empty: continue
        by_lv = sub.groupby('level')['auc'].mean().reindex(LEVELS)
        ax.plot(LEVELS, by_lv.values, marker='o', label=ds, color=color,
                linewidth=2, markersize=7)
    # Global
    ax.plot(LEVELS, agg_s2['auc_mean'].values, marker='D', color='black',
            linewidth=2.5, markersize=9, label='Global', zorder=5, linestyle='--')
    ax.fill_between(LEVELS,
                    agg_s2['auc_mean'] - agg_s2['auc_std'],
                    agg_s2['auc_mean'] + agg_s2['auc_std'],
                    alpha=0.15, color='black')
    ax.axhline(0.5, color='red', linestyle=':', linewidth=1, alpha=0.7)
    ax.set_xlabel('Nível de canais')
    ax.set_ylabel('AUC médio')
    ax.set_title('AUC por dataset e nível de canais')
    ax.legend(fontsize=9)
    ax.set_ylim(0.3, 1.05)
    # Anotação: número de canais
    for i, lv in enumerate(LEVELS):
        ax.annotate(f'{N_CH[lv]}ch', xy=(i, 0.32), ha='center',
                    fontsize=8, color='gray')

    # ── Subplot 2: heatmap AUC por dataset × nível ────────────────────────────
    ax = axes[1]
    datasets = [ds for ds in sorted(DS_COLOR) if ds in df_s2['dataset'].unique()]
    heat = pd.DataFrame(
        {lv: [df_s2[(df_s2['dataset']==ds)&(df_s2['level']==lv)]['auc'].mean()
               for ds in datasets]
         for lv in LEVELS},
        index=datasets
    )
    if HAS_SNS:
        sns.heatmap(heat, annot=True, fmt='.3f', cmap='YlOrRd',
                    vmin=0.4, vmax=1.0, ax=ax, linewidths=0.5,
                    cbar_kws={'label': 'AUC médio'})
    else:
        im = ax.imshow(heat.values, cmap='YlOrRd', vmin=0.4, vmax=1.0, aspect='auto')
        ax.set_xticks(range(len(LEVELS))); ax.set_xticklabels(LEVELS)
        ax.set_yticks(range(len(datasets))); ax.set_yticklabels(datasets)
        plt.colorbar(im, ax=ax, label='AUC médio')
        for i in range(len(datasets)):
            for j in range(len(LEVELS)):
                ax.text(j, i, f'{heat.values[i,j]:.3f}', ha='center', va='center', fontsize=9)
    ax.set_title('Heatmap AUC: Dataset × Nível')
    ax.set_xlabel('Nível de canais')

    plt.tight_layout()
    fig.savefig(FIG_DIR / 'etapa2_niveis.png', bbox_inches='tight', dpi=150)
    plt.show()

    best_lv = agg_s2['auc_mean'].idxmax()
    print(f'Melhor nível global: {best_lv} '
          f'(AUC={agg_s2.loc[best_lv,"auc_mean"]:.3f}±{agg_s2.loc[best_lv,"auc_std"]:.3f})')

---
## 5. Etapa 3 — Comparação de modelos

In [ ]:
ALL_MODELS = ['RF', 'XGB', 'SVM', 'LR', 'NB', 'kNN3', 'kNN5', 'kNN7', 'MDC']

if df_s3.empty:
    print('df_s3 vazio — pule esta célula.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    fig.suptitle('Etapa 3 — Comparação de Modelos de Classificação\n'
                 '(janela consenso + nível ótimo por paciente)', fontsize=13, y=1.01)

    # ── Subplot 1: heatmap AUC dataset × modelo ───────────────────────────────
    ax = axes[0]
    datasets = [ds for ds in sorted(DS_COLOR) if ds in df_s3['dataset'].unique()]
    heat_m = pd.DataFrame(
        {m: [df_s3[(df_s3['dataset']==ds)&(df_s3['model']==m)]['auc'].mean()
              for ds in datasets]
         for m in ALL_MODELS},
        index=datasets
    )
    if HAS_SNS:
        sns.heatmap(heat_m, annot=True, fmt='.3f', cmap='RdYlGn',
                    vmin=0.4, vmax=1.0, ax=ax, linewidths=0.5,
                    cbar_kws={'label': 'AUC médio'})
    else:
        im = ax.imshow(heat_m.values, cmap='RdYlGn', vmin=0.4, vmax=1.0, aspect='auto')
        ax.set_xticks(range(len(ALL_MODELS))); ax.set_xticklabels(ALL_MODELS, rotation=45, ha='right')
        ax.set_yticks(range(len(datasets))); ax.set_yticklabels(datasets)
        plt.colorbar(im, ax=ax, label='AUC médio')
        for i in range(len(datasets)):
            for j in range(len(ALL_MODELS)):
                ax.text(j, i, f'{heat_m.values[i,j]:.3f}', ha='center', va='center', fontsize=8)
    ax.set_title('Heatmap AUC: Dataset × Modelo')

    # ── Subplot 2: barplot global com erro ────────────────────────────────────
    ax = axes[1]
    gbl = (df_s3.groupby('model').agg(
               auc_mean=('auc','mean'), auc_std=('auc','std'),
               f1_mean=('f1','mean')
           ).reindex(ALL_MODELS))
    sort_idx = gbl['auc_mean'].fillna(0).argsort()[::-1]
    sorted_models = [ALL_MODELS[i] for i in sort_idx]

    colors_m = plt.cm.RdYlGn(np.linspace(0.2, 0.85, len(sorted_models)))
    bars = ax.bar(range(len(sorted_models)), gbl.loc[sorted_models, 'auc_mean'],
                  color=colors_m, edgecolor='white', linewidth=0.5)
    ax.errorbar(range(len(sorted_models)), gbl.loc[sorted_models, 'auc_mean'],
                yerr=gbl.loc[sorted_models, 'auc_std'],
                fmt='none', color='black', capsize=4, linewidth=1.2)
    for i, (bar, m) in enumerate(zip(bars, sorted_models)):
        v = gbl.loc[m, 'auc_mean']
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.012,
                f'{v:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.axhline(0.5, color='red', linestyle='--', linewidth=1, alpha=0.7, label='Aleatório')
    ax.set_xticks(range(len(sorted_models)))
    ax.set_xticklabels(sorted_models, rotation=30, ha='right')
    ax.set_ylabel('AUC médio global')
    ax.set_title('Ranking global de modelos (AUC ± std)')
    ax.legend(fontsize=9)
    ax.set_ylim(0.3, 1.1)

    plt.tight_layout()
    fig.savefig(FIG_DIR / 'etapa3_modelos.png', bbox_inches='tight', dpi=150)
    plt.show()

    best_m = gbl['auc_mean'].idxmax()
    print(f'Melhor modelo global: {best_m} '
          f'(AUC={gbl.loc[best_m,"auc_mean"]:.3f}±{gbl.loc[best_m,"auc_std"]:.3f})')

---
## 6. Heatmap de desempenho por paciente (Etapa 3)

Visualiza heterogeneidade inter-paciente — crucial para o TCC.

In [ ]:
if df_s3.empty:
    print('df_s3 vazio.')
else:
    # Agrega AUC por paciente e modelo (escolhe o melhor modelo do fold)
    pat_auc = (df_s3.groupby(['dataset','paciente','model'])['auc']
                    .mean().reset_index())

    # Pivot: linhas = pacientes, colunas = modelos
    pivot = pat_auc.pivot_table(index=['dataset','paciente'],
                                columns='model', values='auc')
    pivot = pivot.reindex(columns=[m for m in ALL_MODELS if m in pivot.columns])

    # Ordena por dataset e AUC médio do RF
    pivot['_ds'] = [k[0] for k in pivot.index]
    pivot['_rf'] = pivot.get('RF', 0).fillna(0)
    pivot = pivot.sort_values(['_ds', '_rf'], ascending=[True, False])
    pivot = pivot.drop(columns=['_ds', '_rf'])

    row_labels = [f"{ds[:3]}/{pat}" for ds, pat in pivot.index]

    fig, ax = plt.subplots(figsize=(13, max(6, len(pivot)*0.45)))
    if HAS_SNS:
        sns.heatmap(pivot.values,
                    xticklabels=list(pivot.columns),
                    yticklabels=row_labels,
                    annot=True, fmt='.2f',
                    cmap='RdYlGn', vmin=0.4, vmax=1.0,
                    linewidths=0.3, ax=ax,
                    cbar_kws={'label': 'AUC médio', 'shrink': 0.6})
    else:
        im = ax.imshow(pivot.values, cmap='RdYlGn', vmin=0.4, vmax=1.0, aspect='auto')
        ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(list(pivot.columns))
        ax.set_yticks(range(len(row_labels))); ax.set_yticklabels(row_labels, fontsize=8)
        plt.colorbar(im, ax=ax, label='AUC médio', shrink=0.6)
    ax.set_title('AUC por paciente e modelo — Etapa 3\n(verde=bom, vermelho=ruim, branco=~chance)', pad=15)
    ax.set_xlabel('Modelo')
    ax.set_ylabel('Dataset / Paciente')

    # Linhas divisórias entre datasets
    ds_list = [k[0] for k in pivot.index]
    prev = None
    for i, ds in enumerate(ds_list):
        if ds != prev and prev is not None:
            ax.axhline(i, color='navy', linewidth=2)
        prev = ds

    plt.tight_layout()
    fig.savefig(FIG_DIR / 'heatmap_pacientes.png', bbox_inches='tight', dpi=150)
    plt.show()
    print(f'Figura salva em {FIG_DIR}/heatmap_pacientes.png')

---
## 7. Radar chart — perfil de métricas do melhor modelo

Comparação de AUC, Sensibilidade, Especificidade e 1−FP/h entre datasets.

In [ ]:
if df_s3.empty:
    print('df_s3 vazio.')
else:
    METRICS = ['AUC', 'Sensibilidade', 'Especificidade', '1 - FP/h norm.']

    def get_radar_vals(df_sub):
        auc  = df_sub['auc'].mean()
        sens = df_sub['sensitivity'].mean()
        spec = df_sub['specificity'].mean()
        # Normaliza FP/h: fp/h=0 → 1.0 (ideal), fp/h=5 → 0.0 (mínimo)
        fp_norm = max(0, 1 - df_sub['fp_h'].mean() / 5)
        return [auc, sens, spec, fp_norm]

    # Pega o melhor modelo por AUC global
    best_m = df_s3.groupby('model')['auc'].mean().idxmax()
    df_best = df_s3[df_s3['model']==best_m]

    N = len(METRICS)
    angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
    angles += angles[:1]  # fecha o polígono

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={'polar': True})
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(METRICS, size=11)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(['0.25', '0.50', '0.75', '1.00'], size=8)

    datasets_avail = [ds for ds in sorted(DS_COLOR)
                      if ds in df_best['dataset'].unique()]
    for ds in datasets_avail:
        vals = get_radar_vals(df_best[df_best['dataset']==ds])
        vals += vals[:1]
        ax.plot(angles, vals, linewidth=2, label=ds, color=DS_COLOR[ds])
        ax.fill(angles, vals, alpha=0.12, color=DS_COLOR[ds])

    # Global
    vals_g = get_radar_vals(df_best)
    vals_g += vals_g[:1]
    ax.plot(angles, vals_g, linewidth=2.5, linestyle='--', color='black', label='Global')

    ax.set_title(f'Radar de métricas — {best_m} (Etapa 3)\n'
                 f'1-FP/h norm.: 0=5FP/h, 1=0FP/h', pad=20, fontsize=12)
    ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=10)

    # Linha de chance (AUC=0.5)
    chance = [0.5] * N + [0.5]
    ax.plot(angles, chance, linewidth=1, linestyle=':', color='red', alpha=0.6)

    plt.tight_layout()
    fig.savefig(FIG_DIR / 'radar_metricas.png', bbox_inches='tight', dpi=150)
    plt.show()
    print(f'Figura salva em {FIG_DIR}/radar_metricas.png')

---
## 8. Distribuição do PRE_SEC estimado pelo NB2

Mostra a variabilidade inter-paciente na janela pré-ictal individual.

In [ ]:
pre_est_path = Path('data/preictal_estimate.json')

if not pre_est_path.exists():
    print('preictal_estimate.json não encontrado — pule esta célula.')
else:
    with open(pre_est_path, encoding='utf-8') as f:
        pre_est = json.load(f)

    df_pre = pd.DataFrame(pre_est)
    df_pre = df_pre[df_pre['pre_sec'].notna()].copy()
    df_pre['pre_min'] = df_pre['pre_sec'] / 60

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle('NB2 — Estimativa Individual da Janela Pré-ictal (PELT + K-Means)',
                 fontsize=13, y=1.01)

    # ── Subplot 1: histograma por dataset ─────────────────────────────────────
    ax = axes[0]
    for ds, color in DS_COLOR.items():
        sub = df_pre[df_pre['dataset']==ds]['pre_min']
        if sub.empty: continue
        ax.hist(sub, bins=8, alpha=0.65, label=ds, color=color, edgecolor='white')
    # Linhas das janelas fixas do NB4
    for wlabel, wmin in [('W1=10','10'), ('W2=15','15'), ('W3=20','20'), ('W4=25','25')]:
        ax.axvline(int(wmin), linestyle='--', linewidth=1.2, alpha=0.7,
                   color='gray', label=f'{wlabel}min')
    ax.set_xlabel('PRE_SEC estimado (min)')
    ax.set_ylabel('Nº de pacientes')
    ax.set_title('Distribuição PRE_SEC por dataset')
    ax.legend(fontsize=9)

    # ── Subplot 2: scatter PRE_SEC por paciente, ordenado ─────────────────────
    ax = axes[1]
    df_sorted = df_pre.sort_values(['dataset', 'pre_min'])
    colors_pat = [DS_COLOR.get(ds, 'gray') for ds in df_sorted['dataset']]
    ax.barh(range(len(df_sorted)), df_sorted['pre_min'],
            color=colors_pat, alpha=0.8, edgecolor='white')
    ax.set_yticks(range(len(df_sorted)))
    ax.set_yticklabels([f"{r.dataset[:3]}/{r.paciente}"
                        for _, r in df_sorted.iterrows()], fontsize=8)
    ax.axvline(df_pre['pre_min'].median(), color='black', linestyle='--',
               linewidth=1.5, label=f'Mediana={df_pre["pre_min"].median():.1f}min')
    ax.set_xlabel('PRE_SEC estimado (min)')
    ax.set_title('PRE_SEC individual por paciente')
    ax.legend(fontsize=9)

    # Legenda de datasets
    patches = [mpatches.Patch(color=c, label=ds) for ds,c in DS_COLOR.items()
               if ds in df_sorted['dataset'].values]
    ax.legend(handles=patches, fontsize=9, loc='lower right')

    plt.tight_layout()
    fig.savefig(FIG_DIR / 'nb2_pre_sec.png', bbox_inches='tight', dpi=150)
    plt.show()
    print(f'PRE_SEC: mediana={df_pre["pre_min"].median():.1f}min, '
          f'IQR=[{df_pre["pre_min"].quantile(.25):.1f}, '
          f'{df_pre["pre_min"].quantile(.75):.1f}]min')

---
## 9. Comparação de datasets — resumo visual lado a lado

In [ ]:
if df_s3.empty:
    print('df_s3 vazio.')
else:
    METRICS_NAMES = ['AUC', 'F1', 'Sens.', 'Espec.', 'FP/h']
    METRICS_COLS  = ['auc', 'f1', 'sensitivity', 'specificity', 'fp_h']

    datasets_avail = [ds for ds in sorted(DS_COLOR)
                      if ds in df_s3['dataset'].unique()]

    best_m = df_s3.groupby('model')['auc'].mean().idxmax()
    df_best = df_s3[df_s3['model']==best_m]

    fig, axes = plt.subplots(1, len(METRICS_COLS), figsize=(16, 5), sharey=False)
    fig.suptitle(f'Comparação entre datasets — modelo {best_m} (Etapa 3)',
                 fontsize=13, y=1.01)

    for ax, col, name in zip(axes, METRICS_COLS, METRICS_NAMES):
        means = [df_best[df_best['dataset']==ds][col].mean() for ds in datasets_avail]
        stds  = [df_best[df_best['dataset']==ds][col].std()  for ds in datasets_avail]
        colors_ds = [DS_COLOR[ds] for ds in datasets_avail]

        bars = ax.bar(range(len(datasets_avail)), means, color=colors_ds,
                      edgecolor='white', linewidth=0.5, alpha=0.85)
        ax.errorbar(range(len(datasets_avail)), means, yerr=stds,
                    fmt='none', color='black', capsize=4, linewidth=1.2)
        for bar, m_val in zip(bars, means):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + (max(stds) if stds else 0) + 0.01,
                    f'{m_val:.3f}', ha='center', va='bottom', fontsize=9,
                    fontweight='bold')

        ax.set_xticks(range(len(datasets_avail)))
        ax.set_xticklabels([d[:5] for d in datasets_avail], rotation=20, ha='right')
        ax.set_title(name)
        ax.set_ylabel(name if ax == axes[0] else '')
        if col != 'fp_h':
            ax.set_ylim(0, 1.15)
            if col == 'auc':
                ax.axhline(0.5, color='red', linestyle='--', linewidth=1, alpha=0.6)

    plt.tight_layout()
    fig.savefig(FIG_DIR / 'comparacao_datasets.png', bbox_inches='tight', dpi=150)
    plt.show()

---
## 10. Análise crítica — os resultados são bons?

Contexto da literatura de predição de crises epilépticas.

In [ ]:
if df_s3.empty:
    print('df_s3 vazio.')
else:
    best_m  = df_s3.groupby('model')['auc'].mean().idxmax()
    df_best = df_s3[df_s3['model']==best_m]
    auc_g   = df_best['auc'].mean()
    sens_g  = df_best['sensitivity'].mean()
    spec_g  = df_best['specificity'].mean()
    fp_g    = df_best['fp_h'].mean()

    # Referências da literatura (valores aproximados)
    lit = {
        'Este trabalho': {'AUC': auc_g, 'Sens': sens_g, 'Espec': spec_g, 'FP/h': fp_g},
        'Truong et al. (2018)\n[CNN, CHB-MIT]':    {'AUC': 0.88, 'Sens': 0.817, 'Espec': None, 'FP/h': 0.16},
        'Tsiouris et al. (2018)\n[LSTM, CHB-MIT]': {'AUC': None, 'Sens': 0.875, 'Espec': None, 'FP/h': 0.21},
        'Usman et al. (2021)\n[SVM features]':     {'AUC': 0.82, 'Sens': 0.80,  'Espec': 0.83, 'FP/h': None},
        'Zanetti et al. (2020)\n[WL+SVM, Siena]':  {'AUC': 0.79, 'Sens': 0.73,  'Espec': 0.82, 'FP/h': 0.35},
    }

    fig, axes = plt.subplots(1, 3, figsize=(14, 6))
    fig.suptitle('Posicionamento na Literatura\n(valores aproximados de referência)',
                 fontsize=13, y=1.01)

    systems = list(lit.keys())
    colors_lit = ['#E53935'] + ['#9E9E9E'] * (len(systems)-1)  # destaca "Este trabalho"

    for ax, metric, col in zip(axes,
                                ['AUC', 'Sens', 'Espec'],
                                ['AUC', 'Sensibilidade', 'Especificidade']):
        vals = [lit[s].get(metric) for s in systems]
        valid = [(s, v, c) for s, v, c in zip(systems, vals, colors_lit) if v is not None]
        if not valid:
            ax.set_visible(False)
            continue
        svs, vvs, cvs = zip(*valid)
        ax.barh(range(len(svs)), vvs, color=cvs, alpha=0.85, edgecolor='white')
        ax.set_yticks(range(len(svs)))
        ax.set_yticklabels(svs, fontsize=8)
        ax.set_xlabel(col)
        ax.set_title(col)
        ax.set_xlim(0, 1.05)
        if metric == 'AUC':
            ax.axvline(0.5, color='red', linestyle='--', linewidth=1, alpha=0.5)
        for i, v in enumerate(vvs):
            ax.text(v + 0.01, i, f'{v:.2f}', va='center', fontsize=9,
                    fontweight='bold' if svs[i]=='Este trabalho' else 'normal')

    plt.tight_layout()
    fig.savefig(FIG_DIR / 'literatura.png', bbox_inches='tight', dpi=150)
    plt.show()

    print('\n=== ANÁLISE CRÍTICA ===')
    print(f'AUC global ({best_m}): {auc_g:.3f}')
    if auc_g >= 0.75:
        print('✓ Resultado ACIMA da linha de base (>0.75) — competitivo com literatura de features manuais')
    elif auc_g >= 0.65:
        print('⚠ Resultado MODERADO (0.65–0.75) — abaixo de abordagens com deep learning, '
              'mas dentro do esperado para features manuais sem ICA')
    else:
        print('✗ Resultado FRACO (<0.65) — próximo do aleatório em média; melhorias necessárias')
    print(f'  Sens={sens_g:.3f}  Espec={spec_g:.3f}  FP/h={fp_g:.2f}')
    print()
    print('Principais limitações identificadas:')
    print('  1. Sem remoção de artefatos (ICA) — artefatos musculares/oculares contaminam features')
    print('  2. Undersample single-seed (seed=42) — variância alta entre folds')
    print('  3. 323 features sem seleção — SVM/kNN sofrem com maldição da dimensionalidade')
    print('  4. Bug filter_window: ctx_ids_inter perdidos quando W < duração interictal do ctx')
    print('  5. PRE_SEC estimado por PELT pode ser ruidoso em sinais com muitos artefatos')

---
## 11. Proposta de melhoria — Z-score por contexto + múltiplas seeds

Implementação das duas melhorias aprovadas, integrada ao `_train_eval_single`.

In [ ]:
# ── Demonstração conceitual das melhorias ────────────────────────────────────
# (Não executa LOSO completo aqui — veja NB4_melhorias.ipynb ou NB_seizeit2_exp.ipynb)

print('=== MELHORIA 1: Z-score por contexto (NB3 → load_patient_features) ===')
print('''
# Em load_patient_features() do NB4, após carregar:
ZSCORE_BY_CTX = True

if ZSCORE_BY_CTX:
    # Z-normaliza cada contexto individualmente antes de empilhar
    # Remove diferenças de escala inter-sessão (impedância, ganho do amplificador)
    # StandardScaler do fold ainda cuida da escala inter-paciente no treino
    for ctx_id in np.unique(d["ctx_ids_pre"]):
        mask = d["ctx_ids_pre"] == ctx_id
        mu = d["X_pre"][mask].mean(axis=0, keepdims=True)
        sd = d["X_pre"][mask].std(axis=0, keepdims=True) + 1e-10
        d["X_pre"][mask] = (d["X_pre"][mask] - mu) / sd
    for ctx_id in np.unique(d["ctx_ids_inter"]):
        mask = d["ctx_ids_inter"] == ctx_id
        mu = d["X_inter"][mask].mean(axis=0, keepdims=True)
        sd = d["X_inter"][mask].std(axis=0, keepdims=True) + 1e-10
        d["X_inter"][mask] = (d["X_inter"][mask] - mu) / sd
''')

print('=== MELHORIA 2: Múltiplas seeds no undersample ===')
print('''
# Em _train_eval_single(), substitui seed única por média de N_SEEDS execuções:
N_SEEDS  = 5
SEEDS    = [42, 43, 44, 45, 46]

seed_results = []
for seed in SEEDS:
    rng   = np.random.RandomState(seed)
    idx_p = rng.choice(len(Xp_tr), n_min, replace=False)
    idx_i = rng.choice(len(Xi_tr), n_min, replace=False)
    X_train = np.vstack([Xp_tr[idx_p], Xi_tr[idx_i]])
    y_train = np.concatenate([np.ones(n_min), np.zeros(n_min)])
    # ... treina, avalia, guarda métricas ...
    seed_results.append({'auc': auc_s, 'f1': f1_s, ...})

# Média das N_SEEDS execuções = métrica do fold
fold_metrics = {k: np.nanmean([r[k] for r in seed_results]) for k in seed_results[0]}
''')

print('Custo estimado: +5× tempo de execução por fold (aceitável para TCC).')
print('Impacto esperado: redução de ~0.03–0.06 no desvio padrão do AUC.')

---
## 12. Mini-experimento: impacto visual das melhorias

Simulação para ilustrar o efeito de múltiplas seeds no AUC.

In [ ]:
np.random.seed(0)

# Simula distribuição de AUC com 1 seed vs 5 seeds
# (baseado em variância típica de undersample em predição de crises)
n_folds   = 40
true_auc  = 0.68  # AUC verdadeiro hipotético
noise_std = 0.09  # desvio padrão por fold (alto com 1 seed)

auc_1seed = np.clip(np.random.normal(true_auc, noise_std, n_folds), 0, 1)
# Com 5 seeds: médias de 5 amostras → std cai por sqrt(5)
auc_5seeds = np.clip(
    np.array([np.random.normal(true_auc, noise_std, 5).mean() for _ in range(n_folds)]),
    0, 1
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Efeito de Múltiplas Seeds no Undersample\n'
             '(simulação ilustrativa, AUC_verdadeiro=0.68)', fontsize=12)

# Histogramas
ax = axes[0]
ax.hist(auc_1seed,  bins=12, alpha=0.65, color='#E53935', label=f'1 seed  (σ={auc_1seed.std():.3f})',  edgecolor='white')
ax.hist(auc_5seeds, bins=12, alpha=0.65, color='#2196F3', label=f'5 seeds (σ={auc_5seeds.std():.3f})', edgecolor='white')
ax.axvline(true_auc, color='black', linestyle='--', linewidth=2, label=f'AUC verdadeiro={true_auc}')
ax.set_xlabel('AUC por fold'); ax.set_ylabel('Frequência')
ax.set_title('Distribuição de AUC: 1 seed vs 5 seeds')
ax.legend(fontsize=10)

# Comparação de médias e CIs
ax = axes[1]
methods = ['1 seed', '5 seeds']
means   = [auc_1seed.mean(), auc_5seeds.mean()]
stds    = [auc_1seed.std(),  auc_5seeds.std()]
colors  = ['#E53935', '#2196F3']
bars = ax.bar(methods, means, color=colors, alpha=0.85, edgecolor='white', width=0.4)
ax.errorbar(methods, means, yerr=1.96*np.array(stds)/np.sqrt(n_folds),
            fmt='none', color='black', capsize=7, linewidth=2, label='IC 95%')
for bar, m, s in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.02,
            f'{m:.3f}±{s:.3f}', ha='center', fontsize=11, fontweight='bold')
ax.axhline(0.5, color='red', linestyle='--', linewidth=1, alpha=0.6, label='Aleatório')
ax.axhline(true_auc, color='black', linestyle=':', linewidth=1.5, label='AUC verdadeiro')
ax.set_ylabel('AUC médio')
ax.set_title('AUC médio e desvio padrão')
ax.legend(fontsize=10)
ax.set_ylim(0.3, 1.0)

plt.tight_layout()
fig.savefig(FIG_DIR / 'impacto_seeds.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'Redução do std: {auc_1seed.std():.3f} → {auc_5seeds.std():.3f} '
      f'(↓{(1-auc_5seeds.std()/auc_1seed.std())*100:.0f}%)')

---
## 13. Tabela resumo para o TCC

In [ ]:
if df_s3.empty:
    print('df_s3 vazio.')
else:
    ALL_MODELS = ['RF', 'XGB', 'SVM', 'LR', 'NB', 'kNN3', 'kNN5', 'kNN7', 'MDC']
    gbl_tcc = (df_s3.groupby('model').agg(
        AUC_media=('auc','mean'), AUC_std=('auc','std'),
        F1_media=('f1','mean'),
        Sens_media=('sensitivity','mean'),
        Espec_media=('specificity','mean'),
        FP_h_media=('fp_h','mean'),
        N_folds=('fold_ctx','count')
    ).reindex(ALL_MODELS)
     .sort_values('AUC_media', ascending=False))

    # Formata para exibição
    fmt = gbl_tcc.copy()
    fmt['AUC'] = [f"{r:.3f}±{s:.3f}"
                  for r, s in zip(gbl_tcc['AUC_media'], gbl_tcc['AUC_std'])]
    fmt['F1']         = gbl_tcc['F1_media'].map('{:.3f}'.format)
    fmt['Sensib.']    = gbl_tcc['Sens_media'].map('{:.3f}'.format)
    fmt['Especif.']   = gbl_tcc['Espec_media'].map('{:.3f}'.format)
    fmt['FP/h']       = gbl_tcc['FP_h_media'].map('{:.2f}'.format)
    fmt['N']          = gbl_tcc['N_folds'].astype(int)
    fmt.index.name = 'Modelo'

    display_cols = ['AUC', 'F1', 'Sensib.', 'Especif.', 'FP/h', 'N']
    print('\nTabela de resultados para o TCC — Etapa 3 (todos os datasets):')
    print('=' * 72)
    print(fmt[display_cols].to_string())
    print()
    print('Legenda: AUC=área sob a curva ROC, F1=F1-score, Sensib.=sensibilidade,')
    print('         Especif.=especificidade, FP/h=falsas detecções por hora, N=nº folds')

    # Salva como CSV para copiar no LaTeX
    fmt[display_cols].to_csv(FIG_DIR / 'tabela_resultados_tcc.csv')
    print(f'\nCSV salvo em {FIG_DIR}/tabela_resultados_tcc.csv')

---
## 14. Próximos passos recomendados

Prioridade baseada na relação custo/benefício para o TCC:

In [ ]:
roadmap = [
    ('1 — ALTA', 'Corrigir bug filter_window (interseção de ctx_ids)',
     '30min', 'Remove folds inválidos; afeta integridade experimental'),
    ('2 — ALTA', 'Z-score por contexto (NB4, load_patient_features)',
     '1h',   'Remove drift inter-sessão sem reprocessar NB1/NB3'),
    ('3 — ALTA', 'Múltiplas seeds no undersample (N_SEEDS=5)',
     '0h*',  'Reduz variância dos folds; *só aumenta tempo de execução'),
    ('4 — MÉD.', 'Notebook piloto SeizeIT2 (NB_pilot_seizeit2.ipynb)',
     '2h',   'Testa todas as melhorias num único dataset antes de escalar'),
    ('5 — MÉD.', 'SelectKBest(MI, k=60) após StandardScaler',
     '30min', '323→60 features; ajuda SVM e kNN; pode melhorar AUC +0.02-0.05'),
    ('6 — BAIXA', 'Grid search (C/γ SVM, n_est RF)',
     '3h',   'Overhead alto com LOSO aninhado; só vale após melhorias 1-3'),
    ('7 — BAIXA', 'ICA automática (MNE) no NB1',
     '>8h',  'Maior impacto potencial, mas exige reprocessar tudo'),
]

print('ROADMAP DE MELHORIAS PARA O TCC')
print('=' * 80)
print(f'{"Prioridade":<12} {"Melhoria":<45} {"Esforço":<10} Justificativa')
print('-' * 80)
for p, m, e, j in roadmap:
    print(f'{p:<12} {m:<45} {e:<10} {j}')
print()
print('Sugestão imediata: criar NB_pilot_seizeit2.ipynb que faz NB1→NB4')
print('apenas para SeizeIT2, aplicando as melhorias 1-3+5 e comparando')
print('com os resultados do NB4 original para o mesmo dataset.')